### Clone Repository and Checkout Branch
Replace `<REPO_URL>` with your Git repository URL (e.g., `https://github.com/user/repo.git`) and `<BRANCH_NAME>` with the branch you want to checkout.

In [7]:
#@title Git Clone Configuration with Authentication
import os
from google.colab import userdata

REPO_URL = "https://github.com/tarek0026/Reliable-Chest-X-Ray-AI.git" #@param {type:"string"}
BRANCH_NAME = "feature/data-pipeline" #@param {type:"string"}
REPO_DIR = "Reliable-Chest-X-Ray-AI" #@param {type:"string"}

# Option 1: Use a Colab Secret named 'GITHUB_TOKEN' (Recommended)
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    # Option 2: Manual fallback parameter if Secret is not configured
    GITHUB_TOKEN = "" #@param {type:"string"}

# Adjust URL to inject token if provided
if GITHUB_TOKEN:
    authenticated_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
else:
    authenticated_url = REPO_URL

# Clone the repository and checkout the branch
!git clone $authenticated_url $REPO_DIR

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH_NAME
else:
    print(f"Failed to clone repository. If this is a private repo, please add your 'GITHUB_TOKEN' to Google Colab Secrets (the key icon on the left panel) and enable Notebook Access.")

Cloning into 'Reliable-Chest-X-Ray-AI'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 69 (delta 23), reused 56 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 11.36 MiB | 18.94 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/Reliable-Chest-X-Ray-AI/Reliable-Chest-X-Ray-AI
branch 'feature/data-pipeline' set up to track 'origin/feature/data-pipeline'.
Switched to a new branch 'feature/data-pipeline'


### Step 1: Mount Google Drive
Mount your Google Drive to access the compressed images archive.

In [8]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Load and Verify Metadata Splits CSV

In [12]:
import os
import pandas as pd

# Verified path inside nested repository clones
metadata_path = '/content/Reliable-Chest-X-Ray-AI/Reliable-Chest-X-Ray-AI/Data/splits/balanced_20k_metadata.csv'

# Fallback paths check
if not os.path.exists(metadata_path):
    fallback_paths = [
        '/content/Reliable-Chest-X-Ray-AI/Data/splits/balanced_20k_metadata.csv',
        'Data/splits/balanced_20k_metadata.csv'
    ]
    for path in fallback_paths:
        if os.path.exists(path):
            metadata_path = path
            break

if os.path.exists(metadata_path):
    df = pd.read_csv(metadata_path)
    print(f"Metadata loaded successfully from: {metadata_path}")
    display(df.head())
else:
    print(f"Could not find the metadata file anywhere. Please verify the path of the cloned repository.")

Metadata loaded successfully from: /content/Reliable-Chest-X-Ray-AI/Reliable-Chest-X-Ray-AI/Data/splits/balanced_20k_metadata.csv


,image_name,finding_labels,follow_up,patient_id,patient_age,patient_sex,view_position,image_width,image_height,pixel_spacing_x,pixel_spacing_y,label,binary_label,split
0,00004338_003.png,Edema,3,4338,49,M,AP,2500,2048,0.171,0.171,1,Abnormal,train
1,00001922_002.png,No Finding,2,1922,44,M,PA,2992,2991,0.143,0.143,0,Normal,train
2,00025082_006.png,Effusion|Mass,7,25082,60,F,PA,2722,2837,0.143,0.143,1,Abnormal,train
3,00006960_003.png,No Finding,3,6960,16,M,PA,2048,2500,0.171,0.171,0,Normal,train
4,00012094_058.png,Emphysema,58,12094,63,F,AP,2500,2048,0.168,0.168,1,Abnormal,train


### Step 3: Extract, Verify Image Integrity, and Organize Dataset

In [14]:
import tarfile
import io
import os
import glob
from PIL import Image
from tqdm import tqdm
import pandas as pd

# Specifically scan Google Drive for your images_tar split files, ignoring unrelated folders
print("Searching Google Drive for dataset archives...")
search_paths = sorted(glob.glob('/content/drive/MyDrive/**/images_*.tar.gz', recursive=True))

if not search_paths:
    # Fallback to general .tar.gz patterns if specific ones aren't found
    all_tar_gz = glob.glob('/content/drive/MyDrive/**/*.tar.gz', recursive=True)
    search_paths = sorted([p for p in all_tar_gz if 'Classroom' not in p])

if search_paths:
    print(f"Found {len(search_paths)} archives to process:")
    for path in search_paths:
        print(f" - {path}")
else:
    print("No matching archive segments found. Please make sure the shortcut is placed in 'My Drive'.")

output_base_dir = '/content/split_dataset'

# Create folders for healthy and corrupted files
categories = ['train', 'validation', 'test']
for cat in categories:
    os.makedirs(os.path.join(output_base_dir, cat), exist_ok=True)
    os.makedirs(os.path.join(output_base_dir, f"{cat}_corrupted"), exist_ok=True)

# Prepare split maps and trackers
metadata_dict = df.set_index(df.columns[0]).to_dict(orient='index')

uncorrupted_metadata = []
corrupted_metadata = []

def check_if_corrupted(file_bytes):
    try:
        img = Image.open(io.BytesIO(file_bytes))
        img.verify() # Verify image integrity
        return False
    except Exception:
        return True

def process_and_split_dataset(archive_paths, dest_base):
    if not archive_paths:
        print("No files to process. Aborting.")
        return

    for archive_path in archive_paths:
        if not os.path.exists(archive_path):
            print(f"Archive segment not found at: {archive_path}. Skipping.")
            continue

        print(f"\nProcessing segment: {os.path.basename(archive_path)}")
        try:
            with tarfile.open(archive_path, 'r:gz') as archive:
                for member in tqdm(archive):
                    if member.isfile():
                        filename = os.path.basename(member.name)

                        if filename in metadata_dict:
                            original_row = metadata_dict[filename].copy()
                            original_row[df.columns[0]] = filename # Re-add filename

                            split_val = str(original_row[df.columns[-1]]).lower()
                            if 'val' in split_val and 'validation' not in split_val:
                                split_val = 'validation'

                            # Extract member content into memory
                            f_obj = archive.extractfile(member)
                            if f_obj is not None:
                                file_data = f_obj.read()

                                is_corrupted = check_if_corrupted(file_data)

                                if is_corrupted:
                                    target_dir = os.path.join(dest_base, f"{split_val}_corrupted")
                                    corrupted_metadata.append(original_row)
                                else:
                                    target_dir = os.path.join(dest_base, split_val)
                                    uncorrupted_metadata.append(original_row)

                                # Write file to the decided location
                                target_filepath = os.path.join(target_dir, filename)
                                with open(target_filepath, 'wb') as out_f:
                                    out_f.write(file_data)
        except Exception as e:
            print(f"Error processing {archive_path}: {e}")

    # Convert lists to DataFrames and save them to CSV
    df_uncorrupted = pd.DataFrame(uncorrupted_metadata)
    df_corrupted = pd.DataFrame(corrupted_metadata)

    df_uncorrupted.to_csv(os.path.join(dest_base, 'uncorrupted_metadata.csv'), index=False)
    df_corrupted.to_csv(os.path.join(dest_base, 'corrupted_metadata.csv'), index=False)

    print(f"\nProcessing completed!")
    print(f"Healthy images sorted: {len(df_uncorrupted)}")
    print(f"Corrupted images identified: {len(df_corrupted)}")

# Run the extraction and verification across all identified image parts
process_and_split_dataset(search_paths, output_base_dir)

Searching Google Drive for dataset archives...
Found 12 archives to process:
 - /content/drive/MyDrive/images_tar/images_01.tar.gz
 - /content/drive/MyDrive/images_tar/images_02.tar.gz
 - /content/drive/MyDrive/images_tar/images_03.tar.gz
 - /content/drive/MyDrive/images_tar/images_04.tar.gz
 - /content/drive/MyDrive/images_tar/images_05.tar.gz
 - /content/drive/MyDrive/images_tar/images_06.tar.gz
 - /content/drive/MyDrive/images_tar/images_07.tar.gz
 - /content/drive/MyDrive/images_tar/images_08.tar.gz
 - /content/drive/MyDrive/images_tar/images_09.tar.gz
 - /content/drive/MyDrive/images_tar/images_10.tar.gz
 - /content/drive/MyDrive/images_tar/images_11.tar.gz
 - /content/drive/MyDrive/images_tar/images_12.tar.gz

Processing segment: images_01.tar.gz


5000it [00:38, 128.31it/s]



Processing segment: images_02.tar.gz


10001it [01:02, 160.12it/s]



Processing segment: images_03.tar.gz


10001it [01:01, 162.96it/s]


... (15 outputs left out)

10001it [01:13, 135.35it/s]



Processing segment: images_12.tar.gz


7122it [00:47, 150.22it/s]



Processing completed!
Healthy images sorted: 20000
Corrupted images identified: 0
